# 05c — Independent Two-Model Pipelines & Sequential Task Cascades (`Sent -> Intent`, `Intent -> Sent`)

**Motivation:** Directly compares our single-backbone joint `MLM+attn` model against non-joint alternatives using the identical splits (`runA`, `runB`), seeds (`42, 43, 44`), and hyperparameters:
1. **Part A — Independent Two-Model Pipelines (`05c_pipeline_2model_plain`, `05c_pipeline_2model_mlm_attn`):**
   - Pairs independent single-task sentiment and single-task intent models ($2\times$ backbone parameters, $2\times$ inference forward passes).
   - Reuses cached Stage-1 validation/test predictions from `05a` & `05b` if present (or trains any missing Stage-1 model inline).
2. **Part B — Sequential Cascade `Sentiment -> Intent` (`05c_cascade_sent_to_intent`):**
   - Stage 1 predicts sentiment ($\hat{y}_S$). Stage 2 conditions the intent classifier via prefix tag `[Sentiment: <label>] <text>`.
   - During training, 20% of training sentiment tags are randomly perturbed (`COND_NOISE_P = 0.20`) so Stage 2 does not over-rely on clean labels; validation and test strictly use Stage-1 predicted sentiment (zero leakage). Tests directly whether predicted `Negative` sentiment rescues rhetorical complaints ending in `?`.
3. **Part C — Sequential Cascade `Intent -> Sentiment` (`05c_cascade_intent_to_sent`):**
   - Stage 1 predicts intent ($\hat{y}_I$); Stage 2 conditions sentiment via `[Intent: <label>] <text>` (`COND_NOISE_P = 0.20` during training; strictly Stage-1 predicted intent on val/test). Tests whether silver-trained intent features distort human sentiment accuracy.
4. **Part D — Parameter Footprint, Inference Latency & Qualitative Diagnostic Check:**
   - Measures parameters, ms/comment latency, and predictions on the qualitative rhetorical-`??` cases (`Run B`, seed `42`).

## 1. Environment & Single Config Cell

In [ ]:
# !pip install -q transformers accelerate
# -- Colab / Repo-Root Setup --------------------------------------------------
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("eval_utils.py"):
    if not os.path.exists("/content/Sinhala-joint-sentiment-and-intent"):
        subprocess.run(
            ["git", "clone", "https://github.com/ThiwankaChanditha/Sinhala-joint-sentiment-and-intent.git", "/content/Sinhala-joint-sentiment-and-intent"],
            check=True,
        )
    os.chdir("/content/Sinhala-joint-sentiment-and-intent")
elif not os.path.exists("eval_utils.py"):
    for _cand in ["..", "../..", "/content/Sinhala-joint-sentiment-and-intent"]:
        if os.path.exists(os.path.join(_cand, "eval_utils.py")):
            os.chdir(_cand)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import gc
import math
import time
import json
import glob
import zipfile
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)

from eval_utils import (
    SENT_CLASSES,
    INTENT_CLASSES,
    SENT_3CLS_INDICES,
    NS,
    NI,
    SEEDS,
    ID_TO_SENT,
    ID_TO_INTENT,
    clean_text,
    log_environment,
    set_all_seeds,
    load_splits,
    macro_f1_present,
    evaluate_and_save,
    summarize_Across_seeds,
)

warnings.filterwarnings("ignore")

# -- Single Config Cell (Identical Hyperparameters to Ablation) ---------------
MODEL_NAME     = "xlm-roberta-base"
MAX_LEN        = 128
MLM_PROB       = 0.15
MLM_EPOCHS     = 3
MLM_BATCH      = 8
MLM_GRAD_ACC   = 2
MLM_LR         = 2e-5

FT_EPOCHS      = 10
FT_BATCH       = 8
FT_GRAD_ACC    = 2
FT_LR          = 2e-5
ENC_LR_MULT    = 0.1
WEIGHT_DECAY   = 0.01
DROPOUT        = 0.3
NUM_ATTN_HEADS = 4
PATIENCE       = 3
COND_NOISE_P   = 0.20   # Train-time conditioning label perturbation for Stage-2 robustness

RUNS_TO_EXEC   = ["runA", "runB"]
SEEDS_TO_EXEC  = SEEDS
NOTEBOOK_NAME  = "05c_two_model_and_cascaded_pipeline"

WORK_DIR       = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
SILVER_CSV     = None
GOLD_CSV       = None
RESULTS_DIR    = os.path.join(WORK_DIR, "results")
PREDS_DIR      = os.path.join(WORK_DIR, "preds")
ENC_DIR        = os.path.join(WORK_DIR, "mlm_encoders")
for _d in (RESULTS_DIR, PREDS_DIR, ENC_DIR):
    os.makedirs(_d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env_info = log_environment()
print("Using device:", DEVICE)

## 2. Shared Encoders, Single-Task Building Blocks & Stage-1 Cache Loader

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
mlm_collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=MLM_PROB)

class SingleTaskDataset(Dataset):
    def __init__(self, texts: list, labels: list):
        enc = tok(texts, truncation=True, max_length=MAX_LEN)
        self.ids = enc["input_ids"]
        self.am  = enc["attention_mask"]
        self.y   = list(labels)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        return {"input_ids": self.ids[idx], "attention_mask": self.am[idx], "label": self.y[idx]}

def collate_single(batch):
    out = tok.pad(
        [{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch],
        return_tensors="pt",
    )
    out["label"] = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    return out

class MeanPooler(nn.Module):
    def __init__(self, hidden: int):
        super().__init__()
        self.norm = nn.LayerNorm(hidden)

    def forward(self, H, mask):
        m = mask.unsqueeze(-1).float()
        return self.norm((H * m).sum(1) / m.sum(1).clamp(min=1.0))

class SingleTaskAttnPooler(nn.Module):
    def __init__(self, hidden: int, heads: int, dropout: float):
        super().__init__()
        self.attn = nn.MultiheadAttention(hidden, heads, dropout=dropout, batch_first=True)
        self.query = nn.Parameter(torch.randn(1, 1, hidden))
        self.norm = nn.LayerNorm(hidden)

    def forward(self, H, mask):
        B = H.size(0)
        out, _ = self.attn(self.query.expand(B, -1, -1), H, H, key_padding_mask=(mask == 0))
        return self.norm(out.squeeze(1))

class SingleTaskClassifier(nn.Module):
    def __init__(self, enc_path: str, num_classes: int, pooling: str = "attn"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(enc_path)
        h = self.encoder.config.hidden_size
        self.pool = SingleTaskAttnPooler(h, NUM_ATTN_HEADS, DROPOUT) if pooling == "attn" else MeanPooler(h)
        self.fc   = nn.Sequential(nn.Linear(h, h // 2), nn.GELU(), nn.Dropout(DROPOUT))
        self.head = nn.Linear(h // 2, num_classes)

    def forward(self, input_ids, attention_mask):
        H = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return self.head(self.fc(self.pool(H, attention_mask)))

def class_weights(ids, K: int) -> torch.Tensor:
    c = np.bincount(ids, minlength=K).astype(float)
    return torch.tensor(np.where(c > 0, c.sum() / (K * np.maximum(c, 1)), 0.0), dtype=torch.float)

@torch.no_grad()
def predict_probs(model, loader) -> np.ndarray:
    model.eval()
    probs = []
    for b in loader:
        logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        probs.append(torch.softmax(logits, dim=-1).cpu())
    return torch.cat(probs, dim=0).numpy()

def get_mlm_encoder(mlm_examples, run: str, seed: int) -> str:
    for c in [os.path.join(ENC_DIR, f"mlm_{run}_seed{seed}"), os.path.join(ENC_DIR, f"mlm_seed{seed}")]:
        if os.path.exists(os.path.join(c, "config.json")):
            return c
    out = os.path.join(ENC_DIR, f"mlm_{run}_seed{seed}")
    os.makedirs(out, exist_ok=True)
    set_all_seeds(seed)
    m = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)
    m.train()
    g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(mlm_examples, batch_size=MLM_BATCH, shuffle=True, collate_fn=mlm_collator, generator=g)
    opt = torch.optim.AdamW(m.parameters(), lr=MLM_LR, weight_decay=WEIGHT_DECAY)
    total = MLM_EPOCHS * math.ceil(len(loader) / MLM_GRAD_ACC)
    sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)
    for ep in range(MLM_EPOCHS):
        tl, n = 0.0, 0
        opt.zero_grad()
        for step, b in enumerate(loader):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            if (b["labels"] != -100).any():
                loss = m(**b).loss
                (loss / MLM_GRAD_ACC).backward()
                tl += loss.item(); n += 1
            if (step + 1) % MLM_GRAD_ACC == 0 or step + 1 == len(loader):
                nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
        print(f"  [MLM {run} seed {seed}] epoch {ep+1}/{MLM_EPOCHS} loss {tl/max(n,1):.4f}")
    m.base_model.save_pretrained(out)
    del m, opt, sched
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

def train_single_task(
    train_texts: list,
    train_y: np.ndarray,
    val_texts: list,
    val_y: np.ndarray,
    test_texts: list,
    num_classes: int,
    enc_path: str,
    pooling: str,
    seed: int,
    tag: str,
    save_dir_ckpt: str = None,
):
    """Train a single-task classifier and return (val_probs, test_probs, best_epoch, best_val_f1)."""
    set_all_seeds(seed)
    train_ds = SingleTaskDataset(train_texts, train_y)
    val_ds   = SingleTaskDataset(val_texts, val_y)
    test_ds  = SingleTaskDataset(test_texts, [0] * len(test_texts))

    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=FT_BATCH, shuffle=True, collate_fn=collate_single, generator=g)
    val_loader   = DataLoader(val_ds,   batch_size=FT_BATCH * 2, shuffle=False, collate_fn=collate_single)
    test_loader  = DataLoader(test_ds,  batch_size=FT_BATCH * 2, shuffle=False, collate_fn=collate_single)

    model = SingleTaskClassifier(enc_path, num_classes=num_classes, pooling=pooling).to(DEVICE)
    W = class_weights(np.asarray(train_y, dtype=np.int64), num_classes).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=W)
    head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
    opt = torch.optim.AdamW(
        [{"params": model.encoder.parameters(), "lr": FT_LR * ENC_LR_MULT},
         {"params": head_params, "lr": FT_LR}],
        weight_decay=WEIGHT_DECAY,
    )
    total = FT_EPOCHS * math.ceil(len(train_loader) / FT_GRAD_ACC)
    sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)

    best_val, best_ep, best_state, bad = -1.0, 0, None, 0
    for ep in range(FT_EPOCHS):
        model.train(); tl = 0.0; opt.zero_grad()
        for step, b in enumerate(train_loader):
            logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
            loss = crit(logits, b["label"].to(DEVICE))
            (loss / FT_GRAD_ACC).backward()
            tl += loss.item()
            if (step + 1) % FT_GRAD_ACC == 0 or step + 1 == len(train_loader):
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
        vp = predict_probs(model, val_loader)
        f1 = macro_f1_present(val_y, vp.argmax(1), num_classes)
        print(f"  [{tag} s{seed}] ep {ep+1:>2} | train {tl/len(train_loader):.4f} | val F1 {f1:.3f}")
        if f1 > best_val:
            best_val, best_ep, bad = f1, ep + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  early stop")
                break

    model.load_state_dict(best_state)
    vp_best = predict_probs(model, val_loader)
    tp_best = predict_probs(model, test_loader)
    if save_dir_ckpt:
        os.makedirs(os.path.dirname(save_dir_ckpt), exist_ok=True)
        torch.save(best_state, save_dir_ckpt)

    del model, opt, sched, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return vp_best, tp_best, best_ep, float(best_val)

def get_or_train_stage1(task: str, variant: str, train_df, val_df, test_df, mlm_examples, run: str, seed: int):
    """Load cached val/test Stage-1 probs from 05a/05b if available, else train & cache inline."""
    assert task in ("sent", "intent")
    model_tag = f"single_task_{task}_{variant}"
    cache_npz = os.path.join(PREDS_DIR, f"stage1_cache_{model_tag}_{run}_{seed}.npz")
    if os.path.exists(cache_npz):
        arr = dict(np.load(cache_npz))
        print(f"[Cache hit] {cache_npz}")
        return arr["val_probs"], arr["test_probs"]

    use_mlm = (variant == "mlm_attn")
    pooling = "attn" if variant == "mlm_attn" else "mean"
    enc_path = get_mlm_encoder(mlm_examples, run=run, seed=seed) if use_mlm else MODEL_NAME
    y_col = "sent_id" if task == "sent" else "intent_id"
    K = NS if task == "sent" else NI
    vp, tp, _, _ = train_single_task(
        train_df["text"].tolist(), train_df[y_col].values,
        val_df["text"].tolist(),   val_df[y_col].values,
        test_df["text"].tolist(),
        num_classes=K, enc_path=enc_path, pooling=pooling,
        seed=seed, tag=f"stage1_{model_tag}_{run}",
    )
    np.savez_compressed(
        cache_npz,
        val_probs=vp.astype(np.float32),
        test_probs=tp.astype(np.float32),
        val_pred=vp.argmax(1).astype(np.int64),
        test_pred=tp.argmax(1).astype(np.int64),
    )
    return vp, tp

## 3. Part A (Two-Model Independent Pipelines), Part B (Sent -> Intent), Part C (Intent -> Sent)

In [ ]:
def corrupt_labels(labels: np.ndarray, K: int, noise_p: float, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed + 1000)
    out = np.asarray(labels, dtype=np.int64).copy()
    mask = rng.random(len(out)) < noise_p
    for idx in np.where(mask)[0]:
        choices = [c for c in range(K) if c != out[idx]]
        out[idx] = rng.choice(choices)
    return out

def format_cond_texts(texts: list, cond_ids: np.ndarray, id_to_label: dict, prefix_name: str) -> list:
    return [f"[{prefix_name}: {id_to_label[int(cid)]}] {txt}" for cid, txt in zip(cond_ids, texts)]

all_summaries = []
diag_records = []

for run in RUNS_TO_EXEC:
    train_df, val_df, test_df = load_splits(run=run, silver_csv=SILVER_CSV, gold_csv=GOLD_CSV)
    mlm_texts = train_df["text"].tolist() + val_df["text"].tolist()
    assert not (set(mlm_texts) & set(test_df["text"])), "Leakage guard!"
    _enc = tok(mlm_texts, truncation=True, max_length=MAX_LEN, return_special_tokens_mask=True)
    mlm_examples = [
        {"input_ids": a, "attention_mask": b, "special_tokens_mask": c}
        for a, b, c in zip(_enc["input_ids"], _enc["attention_mask"], _enc["special_tokens_mask"])
    ]

    p_2m_plain, p_2m_mlm, p_cas_s2i, p_cas_i2s = [], [], [], []

    for seed in SEEDS_TO_EXEC:
        # ---- Stage 1 caches (plain + MLM+attn) ----
        vs_plain, ts_plain = get_or_train_stage1("sent",   "plain",    train_df, val_df, test_df, mlm_examples, run, seed)
        vi_plain, ti_plain = get_or_train_stage1("intent", "plain",    train_df, val_df, test_df, mlm_examples, run, seed)
        vs_mlm,   ts_mlm   = get_or_train_stage1("sent",   "mlm_attn", train_df, val_df, test_df, mlm_examples, run, seed)
        vi_mlm,   ti_mlm   = get_or_train_stage1("intent", "mlm_attn", train_df, val_df, test_df, mlm_examples, run, seed)

        # ---- Part A: Independent Two-Model Pipelines ----
        p_2m_plain.append(evaluate_and_save(
            notebook_name="05c_pipeline_2model_plain",
            model_name="pipeline_2model_plain",
            run=run, seed=seed, test_df=test_df,
            pred_sent_id=ts_plain.argmax(1), pred_intent_id=ti_plain.argmax(1),
            sent_probs=ts_plain, intent_probs=ti_plain,
            extra_meta={"pipeline": "independent_two_model_plain"},
            results_dir=RESULTS_DIR, preds_dir=PREDS_DIR,
        ))
        p_2m_mlm.append(evaluate_and_save(
            notebook_name="05c_pipeline_2model_mlm_attn",
            model_name="pipeline_2model_mlm_attn",
            run=run, seed=seed, test_df=test_df,
            pred_sent_id=ts_mlm.argmax(1), pred_intent_id=ti_mlm.argmax(1),
            sent_probs=ts_mlm, intent_probs=ti_mlm,
            extra_meta={"pipeline": "independent_two_model_mlm_attn"},
            results_dir=RESULTS_DIR, preds_dir=PREDS_DIR,
        ))

        enc_path = get_mlm_encoder(mlm_examples, run=run, seed=seed)

        # ---- Part B: Sequential Cascade Sentiment -> Intent ----
        tr_sent_cond = corrupt_labels(train_df["sent_id"].values, NS, COND_NOISE_P, seed)
        tr_texts_s2i = format_cond_texts(train_df["text"].tolist(), tr_sent_cond, ID_TO_SENT, "Sentiment")
        va_texts_s2i = format_cond_texts(val_df["text"].tolist(),   vs_mlm.argmax(1), ID_TO_SENT, "Sentiment")
        te_texts_s2i = format_cond_texts(test_df["text"].tolist(),  ts_mlm.argmax(1), ID_TO_SENT, "Sentiment")

        ckpt_s2i = os.path.join(PREDS_DIR, f"ckpt_cascade_s2i_{run}_{seed}.pt")
        _, ti_s2i, ep_s2i, vf1_s2i = train_single_task(
            tr_texts_s2i, train_df["intent_id"].values,
            va_texts_s2i, val_df["intent_id"].values,
            te_texts_s2i, num_classes=NI, enc_path=enc_path, pooling="attn",
            seed=seed, tag=f"cascade_S2I_{run}", save_dir_ckpt=ckpt_s2i,
        )

        # Oracle sentiment upper bound (gold sentiment fed to Stage-2 test loader, strictly flagged as oracle diagnostic)
        oracle_meta = None
        if os.path.exists(ckpt_s2i):
            te_oracle = format_cond_texts(test_df["text"].tolist(), test_df["gold_sent_id"].values, ID_TO_SENT, "Sentiment")
            or_ds = SingleTaskDataset(te_oracle, [0] * len(te_oracle))
            or_loader = DataLoader(or_ds, batch_size=FT_BATCH * 2, shuffle=False, collate_fn=collate_single)
            m_or = SingleTaskClassifier(enc_path, num_classes=NI, pooling="attn").to(DEVICE)
            m_or.load_state_dict(torch.load(ckpt_s2i, map_location=DEVICE))
            ti_oracle = predict_probs(m_or, or_loader)
            oracle_meta = {
                "oracle_gold_sent_intent_acc": round(float((test_df["gold_intent_id"].values == ti_oracle.argmax(1)).mean()), 4),
                "oracle_gold_sent_intent_f1": round(macro_f1_present(test_df["gold_intent_id"].values, ti_oracle.argmax(1), NI), 4),
            }
            del m_or; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

        # Check rhetorical question shift on gold complaints ending with '?'
        if seed == 42:
            raw_lower = test_df["raw_text"].astype(str)
            q_neg_mask = (raw_lower.str.contains(r"\?", regex=True)) & (test_df["gold_sentiment"] == "Negative")
            if q_neg_mask.sum() > 0:
                comp_id = INTENT_CLASSES.index("complaint")
                inq_id  = INTENT_CLASSES.index("inquiry")
                diag_records.append({
                    "run": run,
                    "n_q_negative_gold": int(q_neg_mask.sum()),
                    "pipeline_mlm_complaint_hits": int((ti_mlm.argmax(1)[q_neg_mask] == comp_id).sum()),
                    "cascade_s2i_complaint_hits":  int((ti_s2i.argmax(1)[q_neg_mask] == comp_id).sum()),
                    "pipeline_mlm_inquiry_hits":   int((ti_mlm.argmax(1)[q_neg_mask] == inq_id).sum()),
                    "cascade_s2i_inquiry_hits":    int((ti_s2i.argmax(1)[q_neg_mask] == inq_id).sum()),
                })

        p_cas_s2i.append(evaluate_and_save(
            notebook_name="05c_cascade_sent_to_intent",
            model_name="cascade_sent_to_intent",
            run=run, seed=seed, test_df=test_df,
            pred_sent_id=ts_mlm.argmax(1), pred_intent_id=ti_s2i.argmax(1),
            sent_probs=ts_mlm, intent_probs=ti_s2i,
            extra_meta={
                "cascade": "sentiment_to_intent",
                "cond_noise_p": COND_NOISE_P,
                "best_epoch": ep_s2i,
                "best_val_intent_f1": vf1_s2i,
                "oracle_diagnostic": oracle_meta,
            },
            results_dir=RESULTS_DIR, preds_dir=PREDS_DIR,
        ))

        # ---- Part C: Sequential Cascade Intent -> Sentiment ----
        tr_int_cond  = corrupt_labels(train_df["intent_id"].values, NI, COND_NOISE_P, seed)
        tr_texts_i2s = format_cond_texts(train_df["text"].tolist(), tr_int_cond,    ID_TO_INTENT, "Intent")
        va_texts_i2s = format_cond_texts(val_df["text"].tolist(),   vi_mlm.argmax(1), ID_TO_INTENT, "Intent")
        te_texts_i2s = format_cond_texts(test_df["text"].tolist(),  ti_mlm.argmax(1), ID_TO_INTENT, "Intent")

        _, ts_i2s, ep_i2s, vf1_i2s = train_single_task(
            tr_texts_i2s, train_df["sent_id"].values,
            va_texts_i2s, val_df["sent_id"].values,
            te_texts_i2s, num_classes=NS, enc_path=enc_path, pooling="attn",
            seed=seed, tag=f"cascade_I2S_{run}",
        )

        p_cas_i2s.append(evaluate_and_save(
            notebook_name="05c_cascade_intent_to_sent",
            model_name="cascade_intent_to_sent",
            run=run, seed=seed, test_df=test_df,
            pred_sent_id=ts_i2s.argmax(1), pred_intent_id=ti_mlm.argmax(1),
            sent_probs=ts_i2s, intent_probs=ti_mlm,
            extra_meta={
                "cascade": "intent_to_sentiment",
                "cond_noise_p": COND_NOISE_P,
                "best_epoch": ep_i2s,
                "best_val_sent_f1": vf1_i2s,
            },
            results_dir=RESULTS_DIR, preds_dir=PREDS_DIR,
        ))

    for group in (p_2m_plain, p_2m_mlm, p_cas_s2i, p_cas_i2s):
        all_summaries.append(summarize_Across_seeds(group))

df_all = pd.DataFrame(all_summaries)
display(df_all[["run", "model", "sent_acc_str", "sent_f1_3cls_str", "intent_acc_str", "intent_f1_str", "joint_acc_str"]])
if diag_records:
    print("\n--- Rhetorical Question (`?` + Gold Negative Sentiment) Cascade Diagnostic (Seed 42) ---")
    display(pd.DataFrame(diag_records))

with open(os.path.join(RESULTS_DIR, f"{NOTEBOOK_NAME}_summary.json"), "w", encoding="utf-8") as f:
    json.dump({"summaries": all_summaries, "rhetorical_question_diag": diag_records}, f, indent=2)

## 4. Part D — Parameter Count, Latency Benchmark, Qualitative Probe & Packaging

In [ ]:
# Measure exact parameter count & inference latency: Joint (1x backbone) vs Two-Model / Cascade (2x backbones)
class JointRefModel(nn.Module):
    def __init__(self, enc_path: str):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(enc_path)
        h = self.encoder.config.hidden_size
        self.attn = nn.MultiheadAttention(h, NUM_ATTN_HEADS, dropout=DROPOUT, batch_first=True)
        self.qs = nn.Parameter(torch.randn(1, 1, h))
        self.qi = nn.Parameter(torch.randn(1, 1, h))
        self.ns, self.ni_ln = nn.LayerNorm(h), nn.LayerNorm(h)
        self.sfc = nn.Sequential(nn.Linear(h, h // 2), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(h // 2, NS))
        self.ifc = nn.Sequential(nn.Linear(h, h // 2), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(h // 2, NI))

    def forward(self, input_ids, attention_mask):
        H = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        B, kpm = H.size(0), (attention_mask == 0)
        os_, _ = self.attn(self.qs.expand(B, -1, -1), H, H, key_padding_mask=kpm)
        oi_, _ = self.attn(self.qi.expand(B, -1, -1), H, H, key_padding_mask=kpm)
        return self.sfc(self.ns(os_.squeeze(1))), self.ifc(self.ni_ln(oi_.squeeze(1)))

def bench_latency(fn, n_iters: int = 30) -> float:
    for _ in range(5):
        fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_iters):
        fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000.0 / n_iters

dummy = tok(["Data package eka activate une na, salli kapala thiyeddi ai mehema wenne??"] * 16,
            return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN)
d_ids, d_am = dummy["input_ids"].to(DEVICE), dummy["attention_mask"].to(DEVICE)

m_joint = JointRefModel(MODEL_NAME).to(DEVICE).eval()
m_s1    = SingleTaskClassifier(MODEL_NAME, NS, "attn").to(DEVICE).eval()
m_s2    = SingleTaskClassifier(MODEL_NAME, NI, "attn").to(DEVICE).eval()

n_params_joint = sum(p.numel() for p in m_joint.parameters())
n_params_two   = sum(p.numel() for p in m_s1.parameters()) + sum(p.numel() for p in m_s2.parameters())

with torch.no_grad():
    ms_joint = bench_latency(lambda: m_joint(d_ids, d_am)) / 16.0
    ms_two   = bench_latency(lambda: (m_s1(d_ids, d_am), m_s2(d_ids, d_am))) / 16.0

del m_joint, m_s1, m_s2
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

footprint = [
    {"architecture": "Joint MLM+attn (1x encoder)", "params_M": round(n_params_joint / 1e6, 2), "ms_per_comment": round(ms_joint, 3)},
    {"architecture": "Two-Model / Cascade (2x encoders)", "params_M": round(n_params_two / 1e6, 2), "ms_per_comment": round(ms_two, 3)},
]
display(pd.DataFrame(footprint))

# Qualitative check on Table 5.7 diagnostic sentences (Run B, Seed 42)
QUAL_CASES = [
    "Subha aluth awuruddak wewa hemotama! Any offers today?",
    "Data package eka activate une na, salli kapala thiyeddi ai mehema wenne??",
    "Apo mehema service ekak nam minissunta epa wenawa..",
]

pred_csv = os.path.join(PREDS_DIR, "cascade_sent_to_intent_runB_42.csv")
if os.path.exists(pred_csv):
    df_p42 = pd.read_csv(pred_csv)
    print("\n--- Qualitative Inspection Sample from Saved Predictions (cascade_sent_to_intent runB s42) ---")
    sub = df_p42[df_p42["raw_text"].astype(str).str.contains(r"\?", regex=True) & (df_p42["gold_sentiment"] == "Negative")].head(5)
    if len(sub) > 0:
        display(sub[["raw_text", "gold_sentiment", "pred_sentiment", "gold_intent", "pred_intent"]])

ckpt_b42 = os.path.join(PREDS_DIR, "ckpt_cascade_s2i_runB_42.pt")
enc_candidates_b42 = [os.path.join(ENC_DIR, "mlm_runB_seed42"), os.path.join(ENC_DIR, "mlm_seed42")]
enc_b42 = next((p for p in enc_candidates_b42 if os.path.exists(os.path.join(p, "config.json"))), None)
if os.path.exists(ckpt_b42) and enc_b42 is not None:
    m_probe = SingleTaskClassifier(enc_b42, num_classes=NI, pooling="attn").to(DEVICE).eval()
    m_probe.load_state_dict(torch.load(ckpt_b42, map_location=DEVICE))
    comp_id = INTENT_CLASSES.index("complaint")
    inq_id  = INTENT_CLASSES.index("inquiry")
    probe_rows = []
    for raw_txt in QUAL_CASES:
        cleaned = clean_text(raw_txt, run="runB")
        row_out = {"raw_text": raw_txt}
        for cond_lbl in ("Neutral", "Negative"):
            inp = tok([f"[Sentiment: {cond_lbl}] {cleaned}"], return_tensors="pt", truncation=True, max_length=MAX_LEN)
            with torch.no_grad():
                probs = torch.softmax(m_probe(inp["input_ids"].to(DEVICE), inp["attention_mask"].to(DEVICE)), dim=-1).cpu().numpy()[0]
            pred_lbl = ID_TO_INTENT[int(probs.argmax())]
            row_out[f"cond_{cond_lbl}_pred"] = f"{pred_lbl} ({probs.max():.3f})"
            row_out[f"cond_{cond_lbl}_comp_vs_inq"] = f"{probs[comp_id]:.3f} / {probs[inq_id]:.3f}"
        probe_rows.append(row_out)
    del m_probe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("\n--- Counterfactual Sentiment Prefix Probe on Cascade Sent -> Intent (Run B, Seed 42) ---")
    display(pd.DataFrame(probe_rows))

zip_path = os.path.join(WORK_DIR, f"{NOTEBOOK_NAME}_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in (RESULTS_DIR, PREDS_DIR):
        for fp in sorted(glob.glob(os.path.join(folder, "*05c*")) + glob.glob(os.path.join(folder, "*pipeline_2model*")) + glob.glob(os.path.join(folder, "*cascade_*"))):
            if not fp.endswith(".pt"):
                zf.write(fp, arcname=os.path.relpath(fp, WORK_DIR))
print(f"Packaged outputs into: {zip_path}")